# Lesson 01 - 介紹 AI 代理人

歡迎來到 **AI 代理初學者** 課程的第一課！

**AI 代理人** 是一個使用大型語言模型（LLM）作為推理引擎的程式，並且能在現實世界中採取*行動* — 呼叫 API、查詢資料庫或執行程式碼 — 以代表使用者完成目標。

在此筆記本中，你將建立你的第一個代理人：一個推薦度假目的地的 **旅行代理人**。在這過程中你將學習如何：

1. 使用 **Microsoft 代理框架** 連接 Azure AI Foundry 代理服務。
2. 給代理人一個 **工具** — 一個它可以呼叫的純 Python 函數。
3. 執行代理人並檢視其回應。
4. 以逐字元串流的方式取得代理人的回應。


## 設定

在執行此筆記本之前，請確保您已經：

1. **擁有一個已部署聊天模型的 Azure AI Foundry 專案**（例如 `gpt-4o-mini`）。
2. **已使用 Azure CLI 登入** — 在終端機中執行 `az login`。
3. **設定所需的環境變數：**
   - `AZURE_AI_PROJECT_ENDPOINT` — 您的 Azure AI Foundry 專案端點。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您已部署模型的名稱。

以下儲存格會安裝您所需要的 Python 套件。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 建立你的第一個代理人

一個代理人需要兩樣東西：

- **指令**，告訴它*它是誰*以及*如何行事*（系統提示）。
- **工具** — 使用 `@tool` 裝飾的 Python 函式，代理人可以呼叫它們來取得資訊或執行動作。

以下我們定義了一個簡單的工具，回傳熱門度假地點清單。當使用者詢問旅遊建議時，代理人將會使用這個工具。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## 流式回應

為了提供更具互動性的體驗，您可以**串流**代理的回應。代理不必等待完整回覆完成，而是會隨著產生的文字片段即時產出。這在需要即時顯示輸出的聊天介面中特別有用。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

在本課程中，您學到了如何：

- **建立一個提供者**，透過 `AzureAIProjectAgentProvider` 連接到 Azure AI Foundry Agent 服務。
- **使用 `@tool` 裝飾器定義工具**，讓代理能呼叫您的 Python 函式。
- **執行代理** 並傳入使用者訊息，並列印其回應。
- **串流回應**，以取得即時輸出。

在下一課中，我們將更深入探討代理框架，並學習如何賦予代理更強大的工具與多步驟推理能力。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能存在錯誤或不準確之處。原始文件的原語言版本應視為權威來源。對於重要資訊，建議採用專業人工翻譯。我們對因使用本翻譯而產生的任何誤解或誤譯概不負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
